# C11x · Hướng 3 — sàn theo số ứng viên (K1, K4) so với A0, A8, A2 · seed 4–6 · chạy thật

**Cách chạy**
- Runtime → Change runtime type → **A100 GPU** → **Run all**.
- Bị ngắt (hết phiên, mất mạng): kết nối lại rồi **Run all lần nữa**. Run đã xong tự bỏ qua; run dở chạy tiếp từ epoch cuối đã lưu trên Drive (`save_period=1`).

**Giao thức đầy đủ của C11, không rút gọn, không có chế độ demo:**
- YOLO26n-OBB, AI-TOD-R woX đủ 8.426 ảnh train, 24 epoch;
- `ultralytics==8.4.174`, khởi tạo `yolo26n.pt` (COCO), cùng recipe.

**Một lần Run all làm hết:**
1. kiểm tra trước (vài giây sau khi cài);
2. test;
3. dữ liệu;
4. cổng H2;
5. train R0 + 15 run;
6. chấm hai head;
7. kiểm R0;
8. bảng tổng hợp và bootstrap B = 10.000.

Ngay sau epoch đầu tiên, notebook in **tổng thời gian và CU dự kiến**.

**Notebook dừng sớm, có chủ đích, khi:**
1. Ô kiểm tra trước báo lỗi: thiếu A100, thiếu dữ liệu hay code C11 trên Drive, sai phiên bản, thiếu dung lượng.
2. Test hỏng.
3. **Cổng H2 trượt** (preregistration §5). Theo quy tắc đã đăng ký, khi đó **không** train hướng 3. Đây là kết quả nghiên cứu, không phải lỗi.

**Thời gian** [ước lượng: 0,55–0,57 giờ/run ở #23 của C11; chấm ~337 giây/run/head ở #29]:

| Việc | Thời gian |
|---|---|
| Cổng H2, nếu chưa chạy notebook phân tích | ~1 giờ |
| Train 16 run | ~9 giờ |
| Chấm | ~3 giờ |
| Bootstrap | ~3 giờ |

Một phiên Colab thường không đủ dài cho tất cả, nên sẽ phải Run all lại vài lần; mỗi lần chạy tiếp từ chỗ dừng.

## 0. Cấu hình

Đường dẫn giống `C11_colab.ipynb`. Code cũ (`scripts/c11`) đọc từ **Drive**; code mới (`scripts/c11x`) lấy từ
**GitHub** (`BRANCH`). Đổi `BRANCH` thành `main` sau khi PR được merge.

In [ ]:
from pathlib import Path
import sys, json, time, subprocess

DRIVE      = Path("/content/drive/MyDrive/NCKH")
CODE_DIR   = DRIVE / "repo/NCKH/scripts/c11"          # code C11 trên Drive (không có trên GitHub)
DATA_DRIVE = DRIVE / "data"
EXP_DRIVE  = DRIVE / "repo/NCKH/experiments"
LOCAL      = Path("/content/aitodr")
PROTOCOL, VERSION = "woX", "oc"
RUNS_OLD   = EXP_DRIVE / "runs-woX"                    # 12 run thăm dò của C11 (A0/A1/A8/A2 × seed 1–3)
RUNS_NEW   = EXP_DRIVE / "runs-woX-c11x"               # run mới theo preregistration
BRANCH     = "admin/c11-novelty-plan"                  # đổi thành "main" sau khi merge
GH_DIR     = Path("/content/nckh_gh")
PERGT_DIR  = EXP_DRIVE / "c11x-pergt-val"              # bảng theo GT trên val của 12 run cũ (dùng chung 2 notebook)
GATE_FILE  = EXP_DRIVE / "c11x-h2-gate.json"           # kết quả cổng H2 (preregistration §5)

class StopRun(Exception):
    """Dừng có chủ đích, kèm lý do (Run all dừng ở ô này)."""

def today():
    """Ngày lúc ghi file (không cố định từ đầu phiên như biến TODAY của C11)."""
    return time.strftime("%Y-%m-%d")

from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "CPU runtime"

# Tốc độ tiêu CU của A100: đọc trong Colab (biểu tượng RAM/Disk → tài nguyên) và sửa nếu khác. [ước lượng]
CU_PER_HOUR = 5.4
ARMS  = ["A0", "A8", "A2", "K1", "K4"]       # preregistration §4; không đổi sau khi thấy kết quả
SEEDS = [4, 5, 6]
R0_DIR = EXP_DRIVE / "runs-woX-c11x-R0"

In [ ]:
!pip -q install "ultralytics==8.4.174" shapely gdown statsmodels
!rm -rf "{GH_DIR}" && git clone -q --depth 1 -b "{BRANCH}" https://github.com/VoDucNhatku/nckh-group7-tiny-obb "{GH_DIR}"
GH_COMMIT = subprocess.check_output(["git", "-C", str(GH_DIR), "rev-parse", "HEAD"]).decode().strip()
!rm -rf /content/c11 && cp -r "{CODE_DIR}" /content/c11
for p in ("/content/c11", str(GH_DIR / "scripts/c11x")):
    if p not in sys.path:
        sys.path.insert(0, p)
import ultralytics, torch, numpy, platform
ENV = dict(ultralytics=ultralytics.__version__, torch=torch.__version__, numpy=numpy.__version__,
           python=platform.python_version(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           c11x_commit=GH_COMMIT)
print(ENV)
assert ENV["ultralytics"] == "8.4.174"

## 0.1 Kiểm tra trước (vài giây). Có lỗi thì in hết ra rồi dừng.

In [ ]:
import c11x_preflight as pf
warns = pf.run_all(code_dir=CODE_DIR, data_drive=DATA_DRIVE, local=LOCAL, runs_old=RUNS_OLD, exp_drive=EXP_DRIVE,
                   prereg=GH_DIR / "experiments" / "preregistration.md", drive=DRIVE)
for w in warns:
    print("CẢNH BÁO:", w)
print("Kiểm tra trước: OK")
print(subprocess.check_output(["git", "-C", str(GH_DIR), "log", "-1", "--format=preregistration: %H %ci", "--",
                               "experiments/preregistration.md"]).decode())

In [ ]:
# Test cũ (55, code C11 trên Drive) + test mới. Bỏ test train trên CPU (vài phút); test nặng khác chạy vì có GPU.
def run_pytest(cwd, args):
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *args], cwd=str(cwd), capture_output=True, text=True)
    tail = (r.stdout.strip().splitlines() or [r.stderr.strip()])[-1]
    print(cwd, "→", tail)
    if r.returncode != 0:
        print(r.stdout[-6000:], r.stderr[-2000:])
        raise StopRun(f"Test hỏng ở {cwd}. Sửa trước khi chạy tiếp.")
run_pytest("/content/c11", ["test_stal_arms.py", "test_evalkit.py", "test_data_rq2.py", "test_bootstrap.py",
                            "test_fastpath.py"])
run_pytest(GH_DIR / "scripts/c11x", ["tests", "--ignore=tests/test_c11x_train.py", "-p", "no:cacheprovider"])

## 1. Dữ liệu và dataset YOLO (giống hệt C11)

Giải nén nhãn + ảnh woX từ Drive, dựng lại `yolo_woX` bằng `aitodr_data.build_yolo_dataset` với **cùng `seed=0`**,
rồi so với `split_manifest.json` đã lưu ngày 20/10. Lệch thì dừng: mọi so sánh với 12 run cũ cần đúng split cũ.

In [ ]:
LOCAL.mkdir(parents=True, exist_ok=True)
lab_zip = DATA_DRIVE / "labels.zip"
assert lab_zip.exists() and open(lab_zip, "rb").read(2) == b"PK", "labels.zip chưa có trong DATA_DRIVE"
!unzip -q -o "{lab_zip}" -d "{LOCAL}"
LABELS = next(p.parent for p in LOCAL.rglob("trainval") if p.is_dir() and any(p.glob("*.txt")))
IMAGES = LOCAL / "images_raw"
IMAGES.mkdir(exist_ok=True)
for z in ["aitod_wo_xview_trainval_imgs.zip", "aitod_wo_xview_test_imgs.zip"]:
    src = DATA_DRIVE / z
    if src.exists() and not (IMAGES / (z + ".done")).exists():
        !cp "{src}" /content/ && unzip -q -o "/content/{z}" -d "{IMAGES}" && rm "/content/{z}" && touch "{IMAGES}/{z}.done"
    print(z, "OK" if (IMAGES / (z + ".done")).exists() else "CHƯA CÓ")

import aitodr_data as ad
XVIEW = ad.xview_stems()
EXCLUDE = (XVIEW["trainval"] | XVIEW["test"]) if PROTOCOL == "woX" else set()
DS = LOCAL / f"yolo_{PROTOCOL}"
man = ad.build_yolo_dataset(LABELS, IMAGES, DS, version=VERSION, exclude_xview=(PROTOCOL == "woX"),
                            val_frac=0.10, seed=0)
old = json.loads(next(EXP_DRIVE.glob(f"*-dataset-{PROTOCOL}/split_manifest.json")).read_text())
assert man["splits"] == old["splits"], ("split khác ngày 20/10", man["splits"], old["splits"])
DATA_YAML = DS / "aitodr.yaml"
print("split khớp:", man["splits"])

In [ ]:
import c11x_train as ct, c11_pipeline as cp
diff = ct.check_recipe(cp.RECIPE)
if diff:
    raise StopRun(f"RECIPE_C11 khác c11_pipeline.RECIPE trên Drive ở các khoá {diff}. Sửa trước khi train.")
print("recipe khớp C11")

## 2. Cổng H2 (preregistration §5)

- Nếu notebook phân tích đã chạy, dùng kết quả của nó (`c11x-h2-gate.json`).
- Nếu chưa, tính ngay tại đây trên 12 run cũ (khoảng 1 giờ). Kết quả ghi vào cùng thư mục với notebook phân tích, nên không phải làm lại hai lần.

In [ ]:
import c11x_pergt as cg
if GATE_FILE.exists():
    gate = json.loads(GATE_FILE.read_text())
else:
    pg = cg.pergt_for_runs(RUNS_OLD, DATA_YAML, DS / "images" / "val", PERGT_DIR)
    gate = cg.h2_gate(pg)
    GATE_FILE.write_text(json.dumps(gate, indent=2))
print(json.dumps(gate, indent=1, ensure_ascii=False))
if not gate["go"]:
    raise StopRun("Cổng H2 trượt: hệ số log1p(n_pos) không dương với CI loại 0. Theo preregistration §5 không train "
                  "hướng 3; chuyển sang hướng 5. Đây là kết quả, không phải lỗi.")

## 3. Train R0 (A2 seed 1, kiểm tái lập) rồi 5 arm × seed 4–6

- Thứ tự: R0 trước, sau đó seed ngoài, arm trong.
- Sau epoch đầu tiên của phiên, notebook in ước lượng tổng thời gian và CU.

In [ ]:
EPOCHS = ct.RECIPE_C11["epochs"]
JOBS = [("A2", 1, R0_DIR)] + [(a, s, RUNS_NEW) for s in SEEDS for a in ARMS]

def run_dir(a, s, p):
    return Path(p) / ct.run_name(a, s)

def epochs_left():
    return sum(0 if (run_dir(*j) / "c11x_done.json").exists() else EPOCHS - min(ct.epochs_logged(run_dir(*j)), EPOCHS)
               for j in JOBS)

EVALS = [(run_dir("A2", 1, R0_DIR), "e2e")] + [(run_dir(a, s, p), h) for a, s, p in JOBS[1:] for h in ("e2e", "nms")]
n_eval_left = sum(not (rd / f"eval_{h}.json").exists() for rd, h in EVALS)
print(f"còn {epochs_left()} epoch và {n_eval_left} lần chấm")
eta = ct.EtaPrinter(epochs_left, n_eval=n_eval_left, cu_per_hour=CU_PER_HOUR)
for a, s, p in JOBS:
    t0 = time.time()
    rd = ct.train_arm(DATA_YAML, a, s, p, extra_callbacks=eta.callbacks())
    print(rd.name, "xong", f"({(time.time() - t0) / 3600:.2f} giờ trong phiên này)", flush=True)

## 4. Chấm bằng AITODReval gốc (code C11 trên Drive): hai head cho 15 run, head e2e cho R0

In [ ]:
import importlib, aitodr_evalkit as ek, c11_bootstrap as cb
for m in (ek, cp, cb): importlib.reload(m)
TEST_IMAGES = DS / "images" / "test"
for rd, head in EVALS:
    out = rd / f"eval_{head}.json"
    if out.exists():
        continue
    if not (rd / "c11x_done.json").exists():
        raise StopRun(f"{rd.name} chưa train xong; chạy lại ô train.")
    st = cp.predict_and_eval(rd / "weights" / "last.pt", LABELS / "test", TEST_IMAGES, out, head=head,
                             version=VERSION, exclude_stems=EXCLUDE, save_dota_dir=rd / f"dota_{head}")
    print(rd.name, head, {k: round(st[k], 4) for k in ("AP", "AP_vt", "AP_t", "AP_s")}, flush=True)

## 5. Kiểm tái lập R0 (preregistration §6)

- **Khớp** khi |Δ mAP50-95 val ở epoch 24| ≤ 0,003 và |Δ AP test e2e| ≤ 0,002.
- **Không khớp:** vẫn dùng 15 run (cùng một đường code), nhưng không gộp chúng với 12 run cũ, và phải báo độ lệch.

In [ ]:
import pandas as pd
rd0, old0 = run_dir("A2", 1, R0_DIR), RUNS_OLD / "c11_A2_s1"
r0 = {"ap_new": json.loads((rd0 / "eval_e2e.json").read_text())["AP"],
      "ap_old": json.loads((old0 / "eval_e2e.json").read_text())["AP"]}
r0["d_ap"] = r0["ap_new"] - r0["ap_old"]
if (old0 / "results.csv").exists():
    new, old = pd.read_csv(rd0 / "results.csv"), pd.read_csv(old0 / "results.csv")
    new.columns, old.columns = new.columns.str.strip(), old.columns.str.strip()
    col = [c for c in new.columns if "mAP50-95" in c][0]
    r0["d_val_ep24"] = float(new[col].iloc[-1] - old[col].iloc[-1])
    r0["max_abs_d_val_any_epoch"] = float((new[col] - old[col]).abs().max())
r0["match"] = abs(r0["d_ap"]) <= 0.002 and abs(r0.get("d_val_ep24", 0.0)) <= 0.003
(EXP_DRIVE / "c11x-r0-check.json").write_text(json.dumps(r0, indent=2))
print(json.dumps(r0, indent=1))

## 6. Bảng, bootstrap B = 10.000, gộp bất định theo seed

- Chỉ dùng 15 run trong `RUNS_NEW` (seed 4–6); **không** trộn với 12 run thăm dò.
- Cặp xác nhận: K4 − A2 và A2 − A0 (preregistration §4).

In [ ]:
import c11x_results as cr
from aitodr_data import scene_key
OUT = EXP_DRIVE / "c11x-kfloor-results"
OUT.mkdir(parents=True, exist_ok=True)
res = cr.collect_runs(RUNS_NEW)
assert (res.groupby(["head", "arm"]).size() == len(SEEDS)).all(), res.groupby(["head", "arm"]).size()
display(cr.summary_table(res).round(4))
for h in ("e2e", "nms"):
    display(cr.per_seed_table(res, h).round(4))
res.to_csv(OUT / "results_long.csv", index=False)
PAIRS = [("K4", "A2"), ("A2", "A0"), ("K1", "A0"), ("K1", "A8"), ("K4", "A0"), ("A8", "A0")]
KEYS = ("AP_vt", "AP_t", "AP_s", "AP_m", "AP")
rows = []
for head in ("e2e", "nms"):
    f = OUT / f"bootstrap_{head}_B10000.json"
    boot = json.loads(f.read_text()) if f.exists() else cr.run_scene_bootstrap(cb, [RUNS_NEW], head, PAIRS, KEYS, 10_000, scene_key, f)
    for x, y in PAIRS:
        for m in KEYS:
            sc, sd = boot[f"{x}-{y}"][m], cr.paired_seed_deltas(res, head, x, y, m)
            cmb = cr.combine_scene_seed(sc, sd)
            rows.append(dict(head=head, pair=f"{x}-{y}", metric=m, delta=sc["delta"], scene_lo=sc["ci_lo"],
                             scene_hi=sc["ci_hi"], skew=cr.ci_skew(sc), seed_t_lo=sd.get("t_lo"), seed_t_hi=sd.get("t_hi"),
                             seed_min=sd.get("min"), seed_max=sd.get("max"), comb_lo=cmb["lo"], comb_hi=cmb["hi"],
                             comb_excl0=cmb.get("excludes_zero")))
ci = pd.DataFrame(rows); display(ci.round(4)); ci.to_csv(OUT / "ci_scene_seed.csv", index=False)
for rd in sorted(RUNS_NEW.glob("c11x_K*_s*")):
    ks = [json.loads(x) for x in (rd / "c11x_kstats.jsonl").read_text().splitlines()]
    print(rd.name, "thống kê quy tắc K ở epoch cuối:", ks[-1])
(OUT / "env.json").write_text(json.dumps(ENV, indent=2))
print("XONG. Kết quả ở", OUT)